# Data Cleaning and Feature Engineering


In [1]:

import os
os.chdir('/Users/ac/Desktop/postharvest-iq')
print("Working directory:", os.getcwd())

Working directory: /Users/ac/Desktop/postharvest-iq


In [3]:
import os
import numpy as np
import pandas as pd
import pickle
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to MySQL successfully")

Connected to MySQL successfully


In [4]:
# Load cereal prices — Northern markets + Southern reference markets
prices = pd.read_sql("""
    SELECT p.date, p.market, p.market_id, p.commodity,
           p.price, p.pricetype, p.currency,
           m.latitude, m.longitude, m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
""", engine)

prices['date'] = pd.to_datetime(prices['date'])
prices['year'] = prices['date'].dt.year
prices['month'] = prices['date'].dt.month

print(f"Total records: {len(prices)}")
print(f"Markets: {prices['market'].unique()}")
print(f"Commodities: {prices['commodity'].unique()}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
print(f"Missing values: {prices.isnull().sum().sum()}")
print()
print("Records per market per commodity:")
print(prices.groupby(['market','commodity']).size().to_string())

Total records: 1770
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2006-01-15 00:00:00 to 2023-07-15 00:00:00
Missing values: 0

Records per market per commodity:
market    commodity
Bolga     Maize        108
          Millet        78
          Sorghum       71
Kumasi    Maize        206
          Millet       188
          Sorghum      127
Tamale    Maize        100
          Millet       129
          Sorghum       76
Techiman  Maize        163
          Millet       195
          Sorghum      118
Wa        Maize         67
          Millet        74
          Sorghum       70


In [5]:
# Load exchange rates — annual values only, clean duplicates
fx = pd.read_sql("""
    SELECT Year, Value as exchange_rate
    FROM ghana_exchange_rates
    WHERE Months = 'Annual value'
    ORDER BY Year
""", engine)

fx['Year'] = fx['Year'].astype(int)

# Keep only new Ghana cedis — filter out old currency (values above 100)
fx_clean = fx[fx['exchange_rate'] < 100].copy()
fx_clean = fx_clean.groupby('Year')['exchange_rate'].mean().reset_index()

print(f"Exchange rate records: {len(fx_clean)}")
print(fx_clean[fx_clean['Year'] >= 2006])

# Merge on year
prices = prices.merge(fx_clean, left_on='year', right_on='Year', how='left')
prices.drop(columns=['Year'], inplace=True)

# Remove duplicates created by merge
prices = prices.drop_duplicates(
    subset=['date', 'market', 'commodity']
).reset_index(drop=True)

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing exchange rates: {prices['exchange_rate'].isnull().sum()}")

Exchange rate records: 56
    Year  exchange_rate
36  2006       0.915107
37  2007       0.932619
38  2008       1.052275
39  2009       1.404967
40  2010       1.429983
41  2011       1.520625
42  2012       1.824867
43  2013       1.981350
44  2014       2.896575
45  2015       3.714642
46  2016       3.909817
47  2017       4.350533
48  2018       4.585325
49  2019       5.217367
50  2020       5.595708
51  2021       5.805700
52  2022       8.272400
53  2023      11.020409
54  2024      14.181942
55  2025      12.527120

After merge: 1770 rows
Missing exchange rates: 0


In [6]:
# Load producer prices — millet and sorghum only (maize not available for Ghana)
producer = pd.read_sql("""
    SELECT Year, Item, Value as producer_price_index
    FROM fao_producer_prices
    WHERE Item IN ('Maize', 'Millet', 'Sorghum')
    AND Months = 'Annual value'
    ORDER BY Item, Year
""", engine)

producer['Year'] = producer['Year'].astype(int)

# Remove old currency outliers
producer_clean = producer[
    producer['producer_price_index'] < 10000
].copy()

# One value per year per item
producer_clean = producer_clean.groupby(
    ['Year', 'Item']
)['producer_price_index'].mean().reset_index()

# Pivot so each crop has its own column
producer_pivot = producer_clean.pivot(
    index='Year', columns='Item', 
    values='producer_price_index'
).reset_index()
producer_pivot.columns.name = None

# Maize not available — use average of millet and sorghum as proxy
producer_pivot['Maize'] = (
    producer_pivot['Millet'] + producer_pivot['Sorghum']
) / 2

print(f"Producer price records: {len(producer_pivot)}")
print(producer_pivot[producer_pivot['Year'] >= 2006])

# Merge onto prices
def get_producer_index(row):
    year_data = producer_pivot[producer_pivot['Year'] == row['year']]
    if len(year_data) == 0:
        return None
    return year_data[row['commodity']].values[0]

prices['producer_price_index'] = prices.apply(
    get_producer_index, axis=1
)

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing producer index: {prices['producer_price_index'].isnull().sum()}")

Producer price records: 35
    Year       Millet    Sorghum        Maize
15  2006   311.713333   239.8700   275.791667
16  2007   286.790000   233.2600   260.025000
17  2008   489.362500   408.6575   449.010000
18  2009   552.125000   457.2650   504.695000
19  2010   559.857500   464.7325   512.295000
20  2011   608.145000   521.5650   564.855000
21  2012    61.740000    65.2200    63.480000
22  2013    76.320000    78.8100    77.565000
23  2014    86.070000    88.9900    87.530000
24  2015   102.770000   101.3000   102.035000
25  2016  1113.040000   877.1775   995.108750
26  2017   775.507500   569.8950   672.701250
27  2018   999.647500   826.9850   913.316250
28  2019   978.195000   757.3775   867.786250
29  2020  1152.955000   979.8625  1066.408750
30  2021  1837.755000  1483.3600  1660.557500
31  2022  2633.450000  2106.7750  2370.112500
32  2023   364.970000   351.4600   358.215000
33  2024   384.780000   374.6000   379.690000
34  2025   475.630000   455.1700   465.400000

After 

In [7]:
# Remove outliers per market per commodity using IQR method
print(f"Records before outlier removal: {len(prices)}")

def remove_outliers(group):
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    return group[
        (group['price'] >= lower) &
        (group['price'] <= upper)
    ]

prices = prices.groupby(
    ['market', 'commodity'], group_keys=False
).apply(remove_outliers).reset_index(drop=True)

print(f"Records after outlier removal: {len(prices)}")
print(f"Records removed: {1770 - len(prices)}")
print()
print("Records per market per commodity after cleaning:")
print(prices.groupby(['market','commodity']).size().to_string())

Records before outlier removal: 1770
Records after outlier removal: 1673
Records removed: 97

Records per market per commodity after cleaning:
market    commodity
Bolga     Maize        102
          Millet        74
          Sorghum       69
Kumasi    Maize        195
          Millet       181
          Sorghum      121
Tamale    Maize         98
          Millet       119
          Sorghum       70
Techiman  Maize        155
          Millet       182
          Sorghum      111
Wa        Maize         61
          Millet        68
          Sorghum       67


/var/folders/68/cg8hvl455rngc_wr5vs3wmsh0000gn/T/ipykernel_1813/3747343585.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(remove_outliers).reset_index(drop=True)


In [8]:
# Normalise price per market per commodity + add time features
scalers = {}
prices['price_scaled'] = 0.0

for (market, commodity), group in prices.groupby(['market', 'commodity']):
    scaler = MinMaxScaler()
    idx = group.index
    prices.loc[idx, 'price_scaled'] = scaler.fit_transform(
        prices.loc[idx, ['price']]
    ).flatten()
    scalers[f"{commodity}_{market}"] = scaler

# Add time features
prices['month_sin'] = np.sin(2 * np.pi * prices['month'] / 12)
prices['month_cos'] = np.cos(2 * np.pi * prices['month'] / 12)

print(f"Scalers created: {len(scalers)}")
print(f"Scaler keys: {list(scalers.keys())}")
print(f"\nPrice scaled range: {prices['price_scaled'].min():.2f} to {prices['price_scaled'].max():.2f}")
print(f"Month sin range: {prices['month_sin'].min():.2f} to {prices['month_sin'].max():.2f}")
print(f"Month cos range: {prices['month_cos'].min():.2f} to {prices['month_cos'].max():.2f}")

Scalers created: 15
Scaler keys: ['Maize_Bolga', 'Millet_Bolga', 'Sorghum_Bolga', 'Maize_Kumasi', 'Millet_Kumasi', 'Sorghum_Kumasi', 'Maize_Tamale', 'Millet_Tamale', 'Sorghum_Tamale', 'Maize_Techiman', 'Millet_Techiman', 'Sorghum_Techiman', 'Maize_Wa', 'Millet_Wa', 'Sorghum_Wa']

Price scaled range: 0.00 to 1.00
Month sin range: -1.00 to 1.00
Month cos range: -1.00 to 1.00


In [10]:
# Normalise exchange rate and producer price index
fx_scaler = MinMaxScaler()
prices['fx_scaled'] = fx_scaler.fit_transform(
    prices[['exchange_rate']]
).flatten()
scalers['exchange_rate'] = fx_scaler

pp_scaler = MinMaxScaler()
prices['spread_scaled'] = pp_scaler.fit_transform(
    prices[['producer_price_index']]
).flatten()
scalers['producer_price_index'] = pp_scaler

print(f"Total scalers: {len(scalers)}")
print(f"fx_scaled range: {prices['fx_scaled'].min():.2f} to {prices['fx_scaled'].max():.2f}")
print(f"spread_scaled range: {prices['spread_scaled'].min():.2f} to {prices['spread_scaled'].max():.2f}")

Total scalers: 17
fx_scaled range: 0.00 to 1.00
spread_scaled range: 0.00 to 1.00


In [11]:
# Save scalers
import os
os.makedirs('app/ml/models', exist_ok=True)

with open('app/ml/models/price_scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print(f"Scalers saved: {len(scalers)}")

# Select final columns and save clean CSV
final_columns = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'price_scaled', 'month_sin', 'month_cos',
    'fx_scaled', 'spread_scaled'
]

df_final = prices[final_columns].copy()
df_final = df_final.dropna()

os.makedirs('data/processed', exist_ok=True)
df_final.to_csv('data/processed/cereals_merged_clean.csv', index=False)

# Final verification

print(f"Total rows: {len(df_final)}")
print(f"Columns: {list(df_final.columns)}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Markets: {df_final['market'].unique()}")
print(f"Commodities: {df_final['commodity'].unique()}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")
print(f"Scalers file: {os.path.exists('app/ml/models/price_scalers.pkl')}")
print(f"CSV file: {os.path.exists('data/processed/cereals_merged_clean.csv')}")

Scalers saved: 17

=== TASK 2 COMPLETE ===
Total rows: 1673
Columns: ['date', 'market', 'commodity', 'price', 'latitude', 'longitude', 'price_scaled', 'month_sin', 'month_cos', 'fx_scaled', 'spread_scaled']
Missing values: 0
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2006-01-15 00:00:00 to 2022-12-15 00:00:00
Scalers file: True
CSV file: True
